In [1]:
import os
import sys

os.chdir("..")
sys.path.insert(0, "src")

In [2]:
from config import settings
import json
from datetime import datetime, timezone, timedelta
import requests
from bank_client import jwt_gen

In [18]:
API_ORIGIN = "https://api.enablebanking.com"
ASPSP_NAME = "Erste Bank"
ASPSP_COUNTRY = "HU"

with open(settings.sessions_info_path, "r") as f:
    session_info = json.load(f)

base_headers = jwt_gen()

# Using the first available account for the following API calls
account_uid = session_info["Erste Bank"]['accounts'][0]['uid']

all_transactions = []

# Retrieving account balances
r = requests.get(f"{API_ORIGIN}/accounts/{account_uid}/balances", headers=base_headers)
if r.status_code == 200:
    print("Balances:")
    print(r.json())
else:
    print(f"Error response {r.status_code}:", r.text)


# Retrieving account transactions (since 90 days ago)
query = {
    "date_from": (datetime.now(timezone.utc) - timedelta(days=90)).date().isoformat(),
}
continuation_key = None
while True:
    if continuation_key:
        query["continuation_key"] = continuation_key
    r = requests.get(
        f"{API_ORIGIN}/accounts/{account_uid}/transactions",
        params=query,
        headers=base_headers,
    )
    if r.status_code == 200:
        resp_data = r.json()
        all_transactions.extend(resp_data["transactions"])
        continuation_key = resp_data.get("continuation_key")
        if not continuation_key:
            print("No continuation key. All transactions were fetched")
            break
        print(f"Going to fetch more transactions with continuation key {continuation_key}")
    else:
        print(f"Error response {r.status_code}:", r.text)

with open("./data/transactions_erste_bank.json", "w") as f:
    json.dump(all_transactions, f, indent=2)

print("All done!")

Balances:
{'balances': [{'name': 'Interim booked balance', 'balance_amount': {'currency': 'HUF', 'amount': '-1673.00'}, 'balance_type': 'ITBD', 'last_change_date_time': '2026-06-22T12:27:08+02:00', 'reference_date': None, 'last_committed_transaction': None}, {'name': 'Interim available balance', 'balance_amount': {'currency': 'HUF', 'amount': '-1673.00'}, 'balance_type': 'ITAV', 'last_change_date_time': '2026-06-22T12:27:08+02:00', 'reference_date': None, 'last_committed_transaction': None}, {'name': 'Balance of type "BLOCKED"', 'balance_amount': {'currency': 'HUF', 'amount': '0.00'}, 'balance_type': 'OTHR', 'last_change_date_time': '2026-06-22T12:27:08+02:00', 'reference_date': None, 'last_committed_transaction': None}]}
No continuation key. All transactions were fetched
All done!


In [2]:
from storage import get_session, Transaction


def get_uncategorized_transactions() -> list[Transaction]:
    with get_session() as session:
        transactions = session.query(Transaction).filter(Transaction.category.is_(None)).all()

    return transactions

In [3]:
transactions_test = get_uncategorized_transactions()

In [ ]:
from itertools import batched

def categorize_batch(transactions: list[Transaction]) -> dict[str, Category]:
    user_message = {}
    
    with open("scr/prompts/categorization_system_prompt.txt", "r") as f:
        system_prompt = f.read()

    for batch in batched(transactions, 50):
        for tr in batch:
            entry = {"remittance_information": tr.remittance_information, 
                     "credit_debit_indicator": tr.credit_debit_indicator,
                     "transaction_code": tr.transaction_code
                     }
            user_message[tr.id] = entry

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        system=system_prompt,
        messages=[
            {"role": "user", "content": user_message}
        ]
    )

    return response.content[0].text

(<storage.Transaction object at 0x114180d70>, <storage.Transaction object at 0x114180da0>, <storage.Transaction object at 0x114180dd0>, <storage.Transaction object at 0x114180e00>, <storage.Transaction object at 0x114180e30>)
(<storage.Transaction object at 0x114180e60>, <storage.Transaction object at 0x114180e90>)


In [5]:
from categorization_agent import get_uncategorized_transactions, categorize_batch

uncat_tr = get_uncategorized_transactions()

uncat_check = {}

for tr in uncat_tr:
    uncat_check[tr.id] = {
        "remittance_information": tr.remittance_information,
        "credit_debit_indicator": tr.credit_debit_indicator,
        "transaction_code": tr.transaction_code,
        "amount": tr.amount
        }
    
uncat_check

{'659e1be709756bbe0cae12d5e403bf9a757b55a0a9ca319c37adc6baca21b753': {'remittance_information': '20260415_d41009f0-04df-4c04-8363-f41ebbe08284',
  'credit_debit_indicator': 'CRDT',
  'transaction_code': 'DOMESTIC_TRANSFER',
  'amount': Decimal('1300.000000')},
 '54a78430091b87484ac1c404ec48a0bc754fba477b697f2fcf069e2fab4f274a': {'remittance_information': '20260415_af2907be-f9f9-411d-8f95-10fee5c47f32',
  'credit_debit_indicator': 'CRDT',
  'transaction_code': 'DOMESTIC_TRANSFER',
  'amount': Decimal('150.000000')},
 '48795fe73e430702be53bfa28d321284229c90ed9db1277d77c78433e67f640c': {'remittance_information': 'késedelmi_kamat_korrekciója_-_jogszabályi_késedelmi_kamatplafon_miatt',
  'credit_debit_indicator': 'CRDT',
  'transaction_code': 'OTHER',
  'amount': Decimal('1.000000')},
 '4de6ca869e12b463e0c9c75f6628039f5be4a7de02abb670e4c0a73c8e212f9b': {'remittance_information': '',
  'credit_debit_indicator': 'DBIT',
  'transaction_code': 'OTHER',
  'amount': Decimal('1990.000000')},
 'cdf